# Sequential transformer training

Run all cells once in a fresh GPU runtime. Authenticate Drive and W&B at the start. The selected notebooks then run sequentially without manual handoffs or external polling. Keep sufficient Colab compute units and Drive space available. Colab termination can still interrupt execution; rerun this launcher to recover.

All training, evaluation, weights and submissions use the existing notebooks and shared recipe. A fresh child process owns each model so GPU memory is released between models. Failure stops the queue. No automatic Kaggle submission is made.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/koalla05/deep-learning-assignment4.git"
REPO_REF = "83399108d6454facef1547e6682b68fc8cfc1e1b"  # verified training code; change for a new experiment
WANDB_PROJECT = "dl-a4-ner"  # None disables W&B
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "models-kyiv-school-of-economics")
DRIVE_ROOT = "/content/drive/MyDrive/dl-a4-ner"

if "google.colab" in sys.modules:
    if "ner.train" in sys.modules:
        raise RuntimeError(
            "Restart the runtime before changing the training checkout or dependencies"
        )
    repo = Path("/content/deep-learning-assignment4")
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(repo)], check=True)
    if subprocess.check_output(
        ["git", "-C", str(repo), "status", "--porcelain"], text=True
    ).strip():
        raise RuntimeError("The Colab checkout has edits; preserve them before selecting REPO_REF")
    subprocess.run(["git", "-C", str(repo), "fetch", "-q", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
    os.chdir(repo)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "transformers/requirements-colab.txt"],
        check=True,
    )
    artifact_root = Path(DRIVE_ROOT)
else:
    if Path.cwd().name == "transformers":
        os.chdir("..")
    artifact_root = Path("transformers")
sys.path.insert(0, os.getcwd())
import getpass
if WANDB_PROJECT and not os.environ.get("NER_SMOKE") and not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = getpass.getpass("W&B API key (hidden): ")

from ner.experiment import setup

OUT, WANDB_PROJECT = setup(artifact_root / "runs", WANDB_PROJECT, WANDB_ENTITY)
print("Repository:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## Choose the queue

The default runs all three encoders. For two, set `MODELS = ["mbert", "mdeberta"]`, for example. Order is preserved. Four two-stage runs are performed per encoder. Logs and queue status are saved in Drive under `sequence/` (`sequence_smoke/` with `NER_SMOKE`). Each encoder saves its fold bundles and ensemble submission in the usual folders.


In [ ]:
MODELS = ["mbert", "mdeberta", "xlmr"]

command = [
    sys.executable, "-u", "transformers/run_sequence.py",
    "--models", *MODELS, "--output", str(OUT),
]
if WANDB_PROJECT:
    command += ["--project", WANDB_PROJECT]
# A separate process group lets Stop interrupt the queue and its descendants.
import signal

process = subprocess.Popen(command, start_new_session=True)
try:
    returncode = process.wait()
except BaseException:
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)
        process.wait()
    raise
if returncode:
    raise subprocess.CalledProcessError(returncode, command)


## After completion or interruption

The cell prints the Drive status-file path. Each model has a separate log. Prefer each encoder’s `ensembles/<encoder>-<hash>/submission.csv` for Kaggle. Run bundles retain weights, tokenizers, probabilities and offsets for future ensemble work.

On runtime termination, mount/authenticate again in a fresh runtime and rerun the same launcher. Verified completed runs are reused; incomplete runs recover from valid checkpoints. The queue status file is an execution record, not a substitute for bundle verification. There is no keep-alive, quota bypass, or automatic new runtime allocation. A hard termination can leave the last status as `running`; consult model bundles and rerun.
